# Semantic + Thematic Music Recommendation System

This notebook shows the full college-project pipeline:

1. Load songs dataset
2. Preprocess lyrics
3. Detect emotion and theme using keyword dictionaries
4. Convert lyrics to Sentence-BERT embeddings
5. Recommend cross-genre songs using semantic + emotional + thematic scoring

**Important:** We are not training BERT from scratch. We use a pretrained Sentence-BERT model and build embeddings for our song lyrics.

In [1]:
# If running inside a fresh notebook environment, install dependencies once:
# !pip install -r requirements.txt

import pandas as pd
import numpy as np

from data_loader import prepare_dataset, dataset_summary
from embedding import load_embedding_model, load_or_create_embeddings
from recommender import MusicRecommender

pd.set_option("display.max_colwidth", 90)

## 1. Load + Preprocess Dataset

The dataset must have these columns:

- `song_name`
- `artist`
- `lyrics`
- `genre`

In [4]:
df = prepare_dataset("data/songs.csv")

print(dataset_summary(df))
df[["song_name", "artist", "genre", "emotion", "theme", "word_count"]].head(10)

{'songs': 1440, 'genres': 11, 'emotions': 6, 'themes': 7}


,song_name,artist,genre,emotion,theme,word_count
0,"""B"" Movie Box Car Blues - Live",The Blues Brothers,blues,sad,self-reflection,221
1,"""E"" Train",Jonny Lang,blues,romantic,self-reflection,147
2,"""Ma"" Rainey's Black Bottom",Ma Rainey,blues,happy,self-reflection,296
3,#Onelove,Web Sheldon,blues,angry,loneliness,290
4,'Cuz I'm Here,Ruthie Foster,blues,sad,loneliness,163
5,'Frisco Town,Memphis Minnie,blues,romantic,self-reflection,448
6,'Tain't Nobody's Biz'ness If I Do,Jay McShann,blues,romantic,struggle,189
7,'Tain't Nobody's Bizness If I Do,Bessie Smith,blues,calm,ambition,183
8,'Til I Reach Home,Robin Trower,blues,romantic,ambition,144
9,'Til My Back Ain't Got No Bone,Albert King,blues,happy,ambition,169


## 2. Emotion + Theme Distribution

Emotion and theme labels are generated using lightweight keyword dictionaries. This is intentionally simple and explainable for a college prototype.

In [7]:
display(df["emotion"].value_counts())
display(df["theme"].value_counts())

emotion
romantic        566
calm            281
sad             216
angry           146
happy           127
motivational    104
Name: count, dtype: int64

theme
self-reflection    677
loneliness         211
loss/grief         190
ambition           139
guilt/regret        85
struggle            82
heartbreak          56
Name: count, dtype: int64

## 3. Load Sentence-BERT and Build Embeddings

This is the main ML step.

We use:

```python
SentenceTransformer('all-MiniLM-L6-v2')
```

This converts each song's lyrics into a dense vector. Similar meanings get vectors close to each other.

In [10]:
model = load_embedding_model()
embeddings = load_or_create_embeddings(df, model, show_progress_bar=True)

print("Embedding matrix shape:", embeddings.shape)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Embedding matrix shape: (1440, 768)


## 4. Create Recommender

Final score:

```text
0.65 * semantic_similarity
+ 0.15 * emotion_match
+ 0.10 * theme_match
+ 0.10 * genre_diff
```

`genre_diff` encourages cross-genre recommendations when meaning matches.

In [13]:
recommender = MusicRecommender(df=df, embeddings=embeddings, model=model)

## 5. Demo: Metal Song About Emptiness/Guilt

Expected behavior: recommend songs from different genres that share sad/guilt/regret meaning.

In [34]:
# ================================
# 🎧 FINAL FIXED PIPELINE (LAST VERSION)
# ================================

import pandas as pd
import numpy as np
from hybrid_model import load_hybrid_checkpoint
from recommender import MusicRecommender

ARTIFACT_DIR = "artifacts"

# 🔹 Load model
model = load_hybrid_checkpoint(f"{ARTIFACT_DIR}/hybrid_model.pt")

# 🔹 Load catalog
song_catalog = pd.read_csv(f"{ARTIFACT_DIR}/hybrid_song_catalog.csv")

# 🔹 Load embeddings
feature_store = np.load(f"{ARTIFACT_DIR}/hybrid_feature_store.npz")
embeddings = feature_store["lyric_embeddings"]

# ================================
# 🔥 FIX 1: ADD SONG NAMES + ARTIST
# ================================

songs_full = pd.read_csv("/Users/suvratsmacbookair/MyLocalFiles/LyricLens/spotify_songs.csv")

# IMPORTANT: align by index (not song_id)
songs_full = songs_full.reset_index(drop=True)

# ensure same length
min_len = min(len(song_catalog), len(songs_full))
song_catalog = song_catalog.iloc[:min_len].copy()
songs_full = songs_full.iloc[:min_len].copy()

# add columns
song_catalog["song_name"] = songs_full["track_name"]
song_catalog["artist"] = songs_full["track_artist"]

print("✅ Added song_name + artist")

# ================================
# 🔹 SANITY CHECK
# ================================
print("Catalog:", song_catalog.shape)
print("Embeddings:", embeddings.shape)

if len(song_catalog) != len(embeddings):
    raise ValueError("Mismatch still exists")

# ================================
# 🔹 CREATE RECOMMENDER
# ================================
recommender = MusicRecommender(
    df=song_catalog,
    embeddings=embeddings,
    model=model
)

# ================================
# 🔍 QUERY (NOW WILL WORK)
# ================================
query_song = song_catalog["song_name"].iloc[10]  # safe pick

print("Using:", query_song)

# ================================
# 🎯 RECOMMEND
# ================================
recommendations = recommender.recommend_by_song(query_song, top_n=5)

# ================================
# 📊 DISPLAY
# ================================
cols = [
    "song_name",
    "artist",
    "genre",
    "emotion",
    "theme",
    "semantic_similarity",
    "final_score",
    "explanation",
]

display(recommendations[cols].reset_index(drop=True))

✅ Added song_name + artist
Catalog: (1440, 10)
Embeddings: (1440, 768)
Using: Una Vaina Loca


,song_name,artist,genre,emotion,theme,semantic_similarity,final_score,explanation
0,SameOldStory,BONES,r&b,romantic,loneliness,0.673270,0.797428,"Recommended because the lyrics carry similar meaning, same theme: loneliness, same emo..."
1,Not Really (Busta K Remix),Sage Charmaine,pop,romantic,loneliness,0.668544,0.794497,"Recommended because the lyrics carry similar meaning, same theme: loneliness, same emo..."
2,Eminence Front,The Who,r&b,romantic,loneliness,0.614616,0.761062,"Recommended because the lyrics carry similar meaning, same theme: loneliness, same emo..."
3,The Sacred,$uicideBoy$,rock,romantic,loneliness,0.611033,0.758841,"Recommended because the lyrics carry similar meaning, same theme: loneliness, same emo..."
4,ATL Hoe,Baby D.,r&b,romantic,loneliness,0.609317,0.757776,"Recommended because the lyrics carry similar meaning, same theme: loneliness, same emo..."


## 6. Try Any Song From Dataset

In [38]:
# ALWAYS pick from same catalog used in recommender
query_song = song_catalog["song_name"].iloc[0]

print("Using:", query_song)

recommendations = recommender.recommend_by_song(query_song, top_n=5)

display(recommendations[[
    "song_name",
    "artist",
    "genre",
    "emotion",
    "theme",
    "final_score",
    "explanation"
]].reset_index(drop=True))

Using: Pangarap


,song_name,artist,genre,emotion,theme,final_score,explanation
0,Girlfriend in a Coma - 2011 Remaster,The Smiths,blues,sad,self-reflection,0.739986,"Recommended because the lyrics carry similar meaning, same theme: self-reflection, sam..."
1,Save Me,Aimee Mann,jazz,sad,self-reflection,0.739888,"Recommended because the lyrics carry similar meaning, same theme: self-reflection, sam..."
2,Ride Wit Me,Nelly,country,sad,self-reflection,0.730642,"Recommended because the lyrics carry similar meaning, same theme: self-reflection, sam..."
3,Afternoon,TENDER,folk,sad,ambition,0.701988,"Recommended because the lyrics carry similar meaning, related theme: self-reflection -..."
4,Drop In the Ocean,OMI,country,sad,self-reflection,0.701985,"Recommended because the lyrics carry similar meaning, same theme: self-reflection, sam..."


## 7. Custom Lyrics Input

This lets you paste your own lyrics and get recommendations from the dataset.

In [49]:
# ================================
# 🎧 FINAL WORKING CUSTOM LYRICS (REAL FIX)
# ================================

from sentence_transformers import SentenceTransformer
import numpy as np
import recommender as rec_module  # IMPORTANT

# ================================
# 🔹 LOAD TEXT MODEL
# ================================
text_model = SentenceTransformer("all-mpnet-base-v2")

# ================================
# 🔹 OVERRIDE EMBEDDING FUNCTION (CORRECT WAY)
# ================================
def custom_embed(model, text):
    emb = text_model.encode([text], normalize_embeddings=True)
    return np.asarray(emb, dtype=np.float32)[0]

# 🔥 PATCH INSIDE recommender (NOT embedding.py)
rec_module.embed_single_text = custom_embed

print("✅ Patch applied correctly")

# ================================
# 🔍 CUSTOM LYRICS
# ================================
custom_lyrics = """
I carry blame through an empty street and whisper sorry to the dark.
Regret keeps replaying in my hollow chest while every mistake feels heavy.
The night is sad and blue, and I want forgiveness after everything is gone.
"""

# ================================
# 🎯 RUN
# ================================
query_context, custom_recs = recommender.recommend_by_custom_lyrics(
    custom_lyrics,
    genre="metal",
    top_n=5,
)

# ================================
# 📊 OUTPUT
# ================================
print("Detected emotion:", query_context.emotion)
print("Detected theme:", query_context.theme)

display(custom_recs[[
    "song_name",
    "artist",
    "genre",
    "emotion",
    "theme",
    "final_score",
    "explanation"
]].reset_index(drop=True))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

✅ Patch applied correctly
Detected emotion: sad
Detected theme: guilt/regret


,song_name,artist,genre,emotion,theme,final_score,explanation
0,Santa Claus Is Coming To Town,The Jackson 5,blues,sad,guilt/regret,0.704529,"Recommended because the lyrics carry similar meaning, same theme: guilt/regret, same e..."
1,TRUE - Single Edit,Spandau Ballet,hip-hop,sad,self-reflection,0.666091,"Recommended because the lyrics carry similar meaning, related theme: guilt/regret -> s..."
2,GATman And Robbin,50 Cent,rock,sad,loneliness,0.649431,"Recommended because the lyrics carry similar meaning, semantic lyric similarity despit..."
3,Miss You Much,Janet Jackson,country,sad,guilt/regret,0.644818,"Recommended because the lyrics carry similar meaning, same theme: guilt/regret, same e..."
4,Loyal To The Game - DJ Quik Remix (Explicit),2Pac,pop,sad,self-reflection,0.642381,"Recommended because the lyrics carry similar meaning, related theme: guilt/regret -> s..."


In [43]:
print(type(model))

<class 'tuple'>


## What To Say In Viva

- We did not train BERT from scratch because that is expensive and unnecessary for a prototype.
- We used pretrained Sentence-BERT as a feature extractor.
- The model converts lyrics into semantic embeddings.
- We combine semantic similarity with emotion/theme labels and a cross-genre bonus.
- This allows recommendations by lyrical meaning instead of audio sound or genre similarity.